# Finance App — Architecture & Feature Breakdown
**Last updated:** 2026-07-03

---

## What this document covers

This notebook documents the architecture and design decisions behind the Finance App. It is structured around `app.py` as the entry point and its supporting modules.

Use it as a reference for *why* each design decision was made, not just *what* the code does.

**Topics covered:**
1. Multi-file / multi-month support (tabs, session state)
2. Bank detection, fingerprinting, split debit/credit mapping
3. Welcome guide (shown before upload, hidden after)
4. Sidebar — category management (add, delete)
5. Category editing — form-based to avoid Streamlit rerun issues
6. Cross-session persistence — `outflow_overrides.json` + `inflow_overrides.json`
7. Cross-month propagation — one change updates all months
8. 50/30/20 budget rule tab — 5-section layout with salary input, projections, and actual spending breakdown
9. Modular structure — `app.py`, `config.py`, `persistence.py`, `categories.py`, `ingestion.py`, and `ui/` subpackage
10. Bucket mutual exclusion — Needs/Wants multiselects filter each other; warning shown for unassigned categories

---
## 1. File uploader — multiple files

```python
uploaded_files = st.file_uploader(
    "Upload your bank statement CSV files — one file per month, multiple files supported",
    type=["csv"],
    accept_multiple_files=True,
)
```

Returns a **list** of file objects (empty list if nothing is uploaded).

Streamlit reruns the entire script on every user interaction. Every time a file is added or removed, the script runs again from top to bottom. The app tracks which files have already been processed to avoid re-reading and re-normalising them on every rerun — see Section 4 (Processing Loop).

---
## 2. Session state — `st.session_state.months`

All loaded month data is stored in a single dictionary:

```python
st.session_state.months = {
    "2026-01": {
        "bank":        "Monzo",
        "month_label": "January 2026",
        "filename":    "monzo_jan.csv",
        "outflow_df":  <DataFrame>,
        "inflow_df":   <DataFrame>,
    },
    "2026-02": {
        "bank":        "Monzo",
        "month_label": "February 2026",
        "filename":    "monzo_feb.csv",
        "outflow_df":  <DataFrame>,
        "inflow_df":   <DataFrame>,
    },
}
```

**Key** → `"YYYY-MM"` string (e.g. `"2026-01"`).
Why this format?
- It sorts chronologically as a plain string without any extra logic (`"2026-01" < "2026-02"`)
- It is derived directly from the dominant month in the normalised DataFrame

**`filename`** is stored so the app can match a month entry back to its source file — needed when cleaning up removed files (Section 5).

---
## 3. New helper — `get_month_info(df)`

```python
def get_month_info(df):
    dominant = pd.to_datetime(df["Date"]).dt.to_period("M").value_counts().idxmax()
    dt = dominant.to_timestamp()
    return dt.strftime("%Y-%m"), dt.strftime("%B %Y")
```

Called immediately after `load_transactions` returns the normalised DataFrame.

| Return value | Example output | Used for |
|---|---|---|
| `month_key` | `"2026-05"` | Dict key in `st.session_state.months` + chronological sort |
| `month_label` | `"May 2026"` | Tab label + success banner text |

**Why dominant month, not the first row's date?**
Monthly bank statements often include a transaction or two from the day before or after the calendar month boundary. Using the first row's date would misidentify a statement for May as April if it starts on 30 April. `.value_counts().idxmax()` finds whichever month appears most in the data — almost always the correct one.

**Why `.to_timestamp()`?**
`pd.to_period("M")` returns a `Period` object representing a month-long range, not a point in time. You can't call `.strftime()` on a Period directly. `.to_timestamp()` converts it to a `Timestamp` (defaulting to the first day of the period), which supports `.strftime()`.

---
## 4. Processing loop — how new files are handled

```python
processed_names = {v["filename"] for v in st.session_state.months.values()}

for uploaded_file in uploaded_files:
    if uploaded_file.name in processed_names:
        continue   # already in session state — skip
    ...
```

`processed_names` is a set of filenames that are already stored in session state. Any file already in that set is skipped entirely. A file is only read, normalised, split, and stored **once** — the first time it appears in the uploader.

### Steps taken for each new file:

| Step | Code | Purpose |
|---|---|---|
| 1 | `pd.read_csv(uploaded_file)` | Read raw CSV |
| 2 | `make_fingerprint(raw_df.columns)` | Create sorted column string |
| 3 | `detect_bank(raw_df.columns)` | Check hardcoded bank signatures |
| 4 | check `bank_mappings.json` | Check previously saved custom mappings |
| 5 | `continue` if still unknown | Let the mapping UI handle it (separate pass) |
| 6 | `load_transactions(file, fmt)` | Normalise + categorise → full DataFrame |
| 7 | split by `Amount` sign | `outflow_df` (< 0) and `inflow_df` (> 0) |
| 8 | store in `st.session_state.months` | Keyed by `month_key` |

---
## 5. Cleanup — removing deselected files

```python
current_names = {f.name for f in uploaded_files}

for mk in list(st.session_state.months.keys()):
    if st.session_state.months[mk]["filename"] not in current_names:
        del st.session_state.months[mk]
```

When the user removes a file from the uploader, Streamlit reruns the script but that file is no longer in `uploaded_files`. This block compares what is currently uploaded against what is stored in session state and deletes any month whose source file has been removed.

This runs **before** the processing loop so stale data never appears in the tabs.

**Why `list(...)` around `.keys()`?**
Python does not allow deleting from a dict while iterating its keys directly — it raises a `RuntimeError`. Converting to a list first creates a snapshot of the keys, making deletion safe.

---
## 6. Mapping UI — unique widget keys for multiple unknown files

`show_mapping_ui` takes a `suffix` parameter so that when two unknown files are uploaded at the same time, each gets its own independent set of widgets:

```python
def show_mapping_ui(raw_df, suffix=""):
    date_col = c1.selectbox("Date column", cols, key=f"map_date_{suffix}")
    desc_col = c2.selectbox("Description column", cols, key=f"map_desc_{suffix}")
    ...

# Called with the filename as the suffix:
show_mapping_ui(raw_df, suffix=uploaded_file.name)
# → key becomes e.g. "map_date_mystery_bank_jan.csv"
```

The filename is used as the suffix because it is unique per file. Without unique keys, Streamlit would throw a `DuplicateWidgetID` error when two unknown files were uploaded simultaneously — the same `key="map_date"` would appear twice in the same rerun.

---
## 7. Success banners — one per loaded month

```python
for month_key, month in sorted_months:
    st.success(f"Detected: **{month['bank']}** — {month['month_label']} loaded")
# Output: "Detected: Monzo — January 2026 loaded"
#         "Detected: Monzo — February 2026 loaded"
```

One green banner per loaded month, displayed above all the tabs in chronological order. The bank name and month label both come from `st.session_state.months`, so the banners update automatically whenever a file is added or removed.

---
## 8. Top-level month tabs

```python
# Sort chronologically — "2026-01" < "2026-02" as plain strings
sorted_months = sorted(st.session_state.months.items())

# Build the tab bar: ["January 2026", "February 2026", "Compare Months", "50/30/20 Rule"]
all_tabs = st.tabs(
    [m["month_label"] for _, m in sorted_months] + ["Compare Months", "50/30/20 Rule"]
)

# Render each month tab
for tab, (month_key, _) in zip(all_tabs[:-2], sorted_months):
    with tab:
        render_month(month_key)

with all_tabs[-2]:
    render_comparison()

with all_tabs[-1]:
    render_5030_tab()
```

`st.tabs()` takes a list of label strings and returns a matching list of tab context managers. We `zip` them with the sorted months so each tab renders the correct month's data.

`render_month(month_key)` lives in `ui/month.py` and receives just the key — it looks up its own data from `st.session_state.months[month_key]`.

---
## 9a. `render_month` — date range filter with guard

```python
month_start = pd.to_datetime(month_key).date()
month_end   = (pd.to_datetime(month_key) + pd.offsets.MonthEnd(0)).date()
guard_min   = month_start - pd.Timedelta(days=2)
guard_max   = month_end   + pd.Timedelta(days=2)

all_dates = [pd.to_datetime(d) for d in
             list(month["outflow_df"]["Date"]) + list(month["inflow_df"]["Date"])]
min_date  = max(min(all_dates).date(), guard_min)
max_date  = min(max(all_dates).date(), guard_max)

date_range = st.date_input(
    "Filter by date range",
    value=(min_date, max_date),
    min_value=guard_min,
    max_value=guard_max,
    key=f"dr_{month_key}",
    help="Narrows the transactions shown. Defaults to the full statement period.",
)
```

**The guard (±2 days)** prevents the date picker from ranging further than 2 days outside the detected calendar month. This catches statements that legitimately start or end a couple of days outside the boundary, while still blocking accidental month overlap.

`pd.offsets.MonthEnd(0)` snaps any date to the last day of its own month — so `pd.to_datetime("2026-05") + MonthEnd(0)` gives `2026-05-31`.

`key=f"dr_{month_key}"` makes each month tab's picker independent — changing May's range does not affect June's.

---
## 9b. `render_month` — filtered views `out` and `inf`

```python
    if len(date_range) == 2:
        s, e = date_range
        out = month["outflow_df"][(month["outflow_df"]["Date"] >= s) &
                                   (month["outflow_df"]["Date"] <= e)]
        inf = month["inflow_df"] [(month["inflow_df"]["Date"]  >= s) &
                                   (month["inflow_df"]["Date"]  <= e)]
    else:
        out = month["outflow_df"]
        inf = month["inflow_df"]
```

`out` and `inf` are **filtered slices** of the stored DataFrames — not copies. Crucially, boolean indexing in pandas **preserves the original integer index** of each row. This matters for category editing (explained next).

The `else` branch is needed because when the user clears the first date in the picker before selecting the second, Streamlit momentarily returns a 1-tuple. Without this guard the `len == 2` unpack would crash.

---
## 9c. `render_month` — category editing (form-based)

### The problem with `st.data_editor` + `SelectboxColumn`

Every time the user opens a dropdown in a `SelectboxColumn`, Streamlit triggers a full script rerun. During that rerun, the editor re-renders from its base data — losing any pending edits that hadn't been saved yet.

### Solution — wrap the editor in `st.form`

`st.form` suppresses reruns for any widget inside it. The script only reruns when the submit button (Apply Changes) is clicked, at which point all pending edits are committed together.

```python
out_orig_index = list(out.index)          # store original df indices BEFORE reset
out_display    = out[["Date", "Description", "Amount", "Category"]].reset_index(drop=True)

with st.form(key=f"out_form_{month_key}"):
    edited_outflow = st.data_editor(
        out_display,
        column_config={
            "Category": st.column_config.SelectboxColumn(
                "Category",
                options=list(st.session_state.categories.keys()),
            ),
        },
        disabled=["Date", "Description", "Amount"],
        key=f"outflow_editor_{month_key}",
    )
    submitted_out = st.form_submit_button("Apply Changes", type="primary")
```

**Why `reset_index(drop=True)`?**
`out` is a filtered slice of `outflow_df`. Boolean indexing in pandas preserves the original non-contiguous index (e.g. rows 0, 3, 7, 12…). Resetting to 0, 1, 2, 3… gives the editor predictable positions. The original indices are saved in `out_orig_index` so the handler can still find the correct row in the full `outflow_df`.

### The stale-render problem — why `st.rerun()` is needed

When Apply Changes fires, Streamlit runs the script once:

1. `out_display` is computed from the current (pre-save) `outflow_df`
2. The form renders with that data
3. The Apply Changes handler updates `outflow_df` in session state
4. The rerun ends — the user sees the stale form, not the updated one

The fix is to force a second rerun immediately after saving, carrying the success message across via session state (a `st.success()` called before `st.rerun()` is discarded when the exception fires):

```python
if submitted_out:
    changed = 0
    for pos, row in edited_outflow.iterrows():
        real_idx = out_orig_index[pos]   # map editor position → outflow_df row
        if row["Category"] == month["outflow_df"].at[real_idx, "Category"]:
            continue
        month["outflow_df"].at[real_idx, "Category"] = row["Category"]
        add_keyword_to_category(row["Category"], row["Description"])
        propagate_category_to_all_months(row["Description"], row["Category"])
        outflow_overrides[make_tx_key(month["outflow_df"].loc[real_idx])] = row["Category"]
        changed += 1
    if changed:
        save_outflow_overrides()
    st.session_state[out_msg_key] = changed   # carry count across rerun
    st.rerun()                                 # re-render with updated outflow_df

# On the next rerun, display the message then clear it
if out_msg_key in st.session_state:
    n = st.session_state[out_msg_key]
    del st.session_state[out_msg_key]
    if n:
        st.success(f"{n} transaction(s) updated.")
    else:
        st.info("No changes detected.")
```

---
## 9d. `render_month` — nested sub-tabs

```python
sub1, sub2 = st.tabs(["Cash Outflow (Payments)", "Cash Inflow (Transfers In)"])

with sub1:
    # outflow data editor, summary table, charts

with sub2:
    # inflow data editor
```

These are `st.tabs` called **inside** a month tab that is itself inside a top-level `st.tabs`. Streamlit supports this nesting natively. Both sub-tabs follow the same form-based pattern described in Section 9c, operating on `out` (outflows) and `inf` (inflows) respectively.

---
## 10. Unique widget keys — why every key has `_{month_key}`

Streamlit requires every interactive widget to have a **globally unique key** within a single rerun. `render_month` is called once per month, so every widget inside it would collide across months unless each key is differentiated with the month key as a suffix.

| Widget | Key (example for Jan 2026) |
|---|---|
| Date range picker | `dr_2026-01` |
| New category text input | `new_cat_2026-01` |
| Add Category button | `add_cat_2026-01` |
| Outflow data editor | `outflow_editor_2026-01` |
| Apply Changes (outflow) | `apply_out_2026-01` |
| Inflow data editor | `inflow_editor_2026-01` |
| Apply Changes (inflow) | `apply_inf_2026-01` |

The month key (`"2026-01"`, `"2026-02"`, …) is the suffix because it is guaranteed unique per tab. This pattern extends to any widget added inside `render_month` in the future — always append `_{month_key}` to the key.

---
## 11. Split debit/credit column support in `show_mapping_ui`

Some banks (e.g. HSBC) export payments and receipts as two separate positive-number columns instead of one signed Amount column. A checkbox in the mapping UI handles this:

```python
split = st.checkbox("Amount is split into two columns (debit/credit)", key=f"map_split_{suffix}")

if split:
    debit_col  = cs1.selectbox("Debit column (money out)", cols, key=f"map_debit_{suffix}")
    credit_col = cs2.selectbox("Credit column (money in)", cols, key=f"map_credit_{suffix}")
else:
    amount_col = c3.selectbox("Amount column", cols, key=f"map_amount_{suffix}")

if st.button("Save & Continue", key=f"map_btn_{suffix}"):
    if split:
        return {"date_col": ..., "description_col": ...,
                "debit_col": debit_col, "credit_col": credit_col}
    return {"date_col": ..., "description_col": ..., "amount_col": amount_col}
```

`normalise_df` already handled `debit_col` / `credit_col` keys:
```python
if "debit_col" in fmt and "credit_col" in fmt:
    debit  = pd.to_numeric(...).fillna(0)
    credit = pd.to_numeric(...).fillna(0)
    df["Amount"] = credit - debit   # credit positive, debit negative
```
No changes to `normalise_df` were needed — the mapping UI just needed to produce the right keys.

---
## 12. Welcome guide — `show_welcome_guide()`

A step-by-step onboarding screen is shown when no CSV files have been uploaded yet. It disappears the moment the first file is added.

```python
def main():
    uploaded_files = st.file_uploader(...)

    if not uploaded_files:
        st.session_state.months = {}
        show_welcome_guide()   # ← only reached when list is empty
        return                 # stops here — nothing else renders
    ...
```

`show_welcome_guide()` renders four columns (Export → Upload → Categorise → Analyse), a supported-banks line, and a learning tip. Because it is placed after `st.file_uploader` but before the `return`, the uploader is always visible at the top regardless of whether the guide is showing.

**Why not place the guide above `st.file_uploader`?**
If the guide appeared above the uploader, the user would have to scroll past all the instructions every time. Placing the guide below means it only occupies space when the page is otherwise empty.

---
## 13. Sidebar — `render_sidebar()` and `delete_category()`

The sidebar gives the user persistent access to category management without it cluttering the main content area.

```python
def render_sidebar():
    with st.sidebar:
        st.header("Categories")
        st.caption("Categories group transactions. The app learns your choices...")

        new_category = st.text_input("Category name", placeholder="e.g. Groceries")
        if st.button("Add Category") and new_category:
            if new_category not in st.session_state.categories:
                st.session_state.categories[new_category] = []
                save_categories()
                st.success(f"{new_category} added.")
            else:
                st.warning(f"{new_category} already exists.")

        for cat in list(st.session_state.categories.keys()):
            col1, col2 = st.columns([4, 1])
            col1.write(cat)
            if cat != "Uncategorized":
                if col2.button("✕", key=f"del_cat_{cat}"):
                    delete_category(cat)
```

**`delete_category`** removes the category from session state, saves to `categories.json`, and resets all transactions in every loaded month that were using it:

```python
def delete_category(cat_name):
    del st.session_state.categories[cat_name]
    save_categories()
    for month in st.session_state.months.values():
        for df_key in ["outflow_df", "inflow_df"]:
            df = month[df_key]
            df.loc[df["Category"] == cat_name, "Category"] = "Uncategorized"
```

**Placement in `main()`** — `render_sidebar()` is called **after** `st.file_uploader`. This is critical: if the sidebar triggers any action (e.g. deleting a category) that causes a rerun, the file uploader has already been rendered and holds its state in the browser. Placing `render_sidebar()` before the file uploader would cause the uploader to lose its files on sidebar-triggered reruns.

---
## 14. Cross-session persistence — `outflow_overrides.json`

### The problem

`outflow_df` lives in `st.session_state`. Pressing F5 clears the session. The user must re-upload their CSV, which re-runs `categorize_transactions`. Keyword matching should restore categories — but only if the description matches a saved keyword exactly. Any edge case (e.g. description slightly different, keyword not saved) means the category is lost.

### Two-layer persistence

| Layer | File | Scope | Applied when |
|---|---|---|---|
| Keywords | `categories.json` | Per description string (all months, all time) | `categorize_transactions` on every new CSV load |
| Per-transaction override | `outflow_overrides.json` | Per exact transaction (`date\|description\|amount`) | `apply_outflow_overrides` immediately after loading |

```python
# Module-level load (runs on every rerun)
if os.path.exists("outflow_overrides.json"):
    with open("outflow_overrides.json", "r") as f:
        outflow_overrides = json.load(f)
else:
    outflow_overrides = {}

def make_tx_key(row):
    return f"{row['Date']}|{row['Description']}|{row['Amount']}"

def apply_outflow_overrides(df):
    for idx, row in df.iterrows():
        key = make_tx_key(row)
        if key in outflow_overrides:
            df.at[idx, "Category"] = outflow_overrides[key]
    return df
```

Called when the month is first stored:
```python
st.session_state.months[month_key] = {
    "outflow_df": apply_outflow_overrides(df[df["Amount"] < 0].copy()),
    "inflow_df":  apply_inflow_overrides(df[df["Amount"] > 0].copy()),
}
```

The override takes precedence over keyword matching because it runs second and explicitly stamps the saved category onto the row, overwriting whatever `categorize_transactions` set.

`inflow_overrides.json` follows the identical pattern and was already in place for inflows because inflow transactions (e.g. two salary transfers with the same description but different amounts) need individual rather than keyword-based categorisation.

---
## 15. Cross-month propagation — `propagate_category_to_all_months()`

### The problem

If "claude.ai" appears in both May and June, and the user categorises it as Subscriptions in June, May still shows Uncategorized. They are separate DataFrames in session state.

### Solution

```python
def propagate_category_to_all_months(description, category):
    desc_lower = description.lower().strip()
    for month_data in st.session_state.months.values():
        df = month_data["outflow_df"]
        mask = df["Description"].str.lower().str.strip() == desc_lower
        df.loc[mask, "Category"] = category
        for _, row in df[mask].iterrows():
            outflow_overrides[make_tx_key(row)] = category
```

Called inside the Apply Changes handler immediately after the single-row update:
```python
month["outflow_df"].at[real_idx, "Category"] = row["Category"]
add_keyword_to_category(row["Category"], row["Description"])
propagate_category_to_all_months(row["Description"], row["Category"])
```

**Three things happen together:**

| Action | What it does | Scope |
|---|---|---|
| `month["outflow_df"].at[real_idx, ...]` | Updates the current row | Current month, in memory |
| `add_keyword_to_category` | Saves description as keyword | All future uploads (`categories.json`) |
| `propagate_category_to_all_months` | Updates every other loaded month's `outflow_df` **and** writes an override key for each matching row | All currently loaded months + persisted to `outflow_overrides.json` |

The loop inside `propagate_category_to_all_months` also writes to `outflow_overrides` — this is what ensures other months stay categorised after an F5 refresh. Without it, the in-memory update to other months' DataFrames would be lost on the next session reset.

---
## 16. 50/30/20 budget rule tab — `render_5030_tab()` (extended)

The tab has been significantly extended from a static educational page into a fully interactive personal finance tool. It lives in `ui/budget_rule.py` and is imported and called by `app.py`:

```python
from ui.budget_rule import render_5030_tab
...
with all_tabs[-1]:
    render_5030_tab()
```

The function has five sections (A–E), plus two helper functions defined at the top that are reused in both the hypothetical projection (Section C) and the actual-rate projection (Section E):

```python
def future_value(pmt, annual_rate, n_months):
    """FV of an ordinary annuity: PMT × [(1+r)^n − 1] / r"""
    if pmt == 0:
        return 0.0
    if annual_rate == 0:
        return float(pmt * n_months)   # avoid division by zero
    r = annual_rate / 12               # monthly rate
    return pmt * ((1 + r) ** n_months - 1) / r

SCENARIOS  = {
    "Cash savings (0%)":            0.00,
    "Easy access savings (3%)":     0.03,
    "Index fund / Stocks ISA (7%)": 0.07,
}
MILESTONES = {1: 12, 5: 60, 10: 120, 20: 240, 40: 480}
MAX_MONTHS = 500

def projection_chart(monthly_pmt, title):
    # Builds a 3-line Plotly chart across all SCENARIOS up to MAX_MONTHS
    # Adds dotted vlines at each milestone and a hline at total contributed

def milestone_table(monthly_pmt):
    # Returns a DataFrame: rows = scenarios, columns = "1 yr" … "40 yr"
    # Values are formatted strings e.g. "£24,000"
```

---

### Section A — The Rule (expander)

The allocation breakdown (Needs / Wants / Savings bullet lists) is inside a collapsed `st.expander` so it does not crowd the top of the tab on repeat visits. The introductory paragraph remains visible above it.

---

### Section B — Budget Calculator

```python
salary = st.number_input(
    "Monthly take-home pay (£)", min_value=0, value=1000, step=50,
    help="Enter your monthly salary or allowance after tax and deductions.",
)

target_20pct = int(salary * 0.20)
slider_max   = max(target_20pct * 2, 200)   # max = 2× target so midpoint = target

monthly_savings = st.slider(
    "Monthly savings (£)",
    min_value=0, max_value=slider_max, value=target_20pct, step=10,
    key="5030_slider",
)
```

- The **slider midpoint is the 20% target** — the user drags it to explore higher or lower rates.
- `max_value = 2 × target` so the 20% figure falls exactly in the middle, giving equal range above and below.
- `max(target_20pct * 2, 200)` prevents a zero max if salary is 0.
- Three `st.metric` cards update live: Needs target (50%), Wants target (30%), and the slider's current % of take-home pay.

---

### Section C — Hypothetical Savings Projection

Uses `monthly_savings` from the slider to build the `projection_chart` and `milestone_table`. The X-axis runs to 500 months (~42 years). Dotted vertical lines mark 1, 5, 10, 20, and 40-year milestones. A dotted horizontal line marks the total amount contributed (no interest), so the user can visually compare interest-earned growth against raw contributions.

This section always renders regardless of whether a CSV has been uploaded — it is a standalone calculator.

---

### Section D — Your Spending vs 50/30/20

Only shown when at least one CSV is uploaded:

```python
if not st.session_state.months:
    st.info("Upload your bank statement CSV files...")
    return
```

**Category bucket assignment — mutual exclusion**

Two multiselect columns let the user assign categories to Needs and Wants. The key behaviour is that each bucket's available options automatically **exclude** whatever the other bucket already has selected, so no category can appear in both at once:

```python
available_cats = [c for c in st.session_state.categories if c != "Uncategorized"]

# Needs options: exclude anything already in Wants (from previous rerun's session state)
needs_options = [c for c in available_cats if c not in st.session_state.get("5030_wants", [])]

col1, col2 = st.columns(2)
with col1:
    st.markdown("**Needs** — essentials")
    needs_cats = st.multiselect(
        "needs_select", needs_options, key="5030_needs",
        label_visibility="collapsed",
    )
with col2:
    # Wants options: exclude whatever was just selected in Needs THIS rerun
    wants_options = [c for c in available_cats if c not in needs_cats]
    st.markdown("**Wants** — lifestyle")
    wants_cats = st.multiselect(
        "wants_select", wants_options, key="5030_wants",
        label_visibility="collapsed",
    )
```

**How the same-rerun update works**

This is the subtle part. When Streamlit reruns (e.g. because the user added "Groceries" to Needs):

1. `st.session_state["5030_needs"]` already contains `["Groceries"]` — Streamlit updates widget keys before the script runs.
2. `needs_options` uses `st.session_state.get("5030_wants", [])` from the *previous* rerun — so Needs correctly excludes the previous Wants selection.
3. The Needs multiselect renders and `needs_cats = ["Groceries"]` is returned.
4. `wants_options` is computed from `needs_cats` (the just-returned value) — so Wants immediately excludes `"Groceries"` in *this same rerun*, with no lag.

The asymmetry (Needs uses previous-rerun session state; Wants uses this-rerun return value) is intentional and is what makes it work correctly in a single pass.

**Unassigned categories warning**

After both multiselects render, any category not in either bucket triggers a warning:

```python
unassigned = [c for c in available_cats if c not in needs_cats and c not in wants_cats]
if unassigned:
    st.warning(
        f"Not yet assigned to Needs or Wants: **{', '.join(unassigned)}**. "
        "Assign all categories for an accurate breakdown."
    )
```

This prompts the user to classify every category — important because unassigned categories are excluded from both the Needs/Wants totals and the derived Savings figure, which would make the breakdown inaccurate.

**Savings is derived, not a bucket**

Savings is not a third multiselect. It is the mathematical leftover from take-home pay after Needs and Wants:

```python
actual_needs   = avg_by_cat[avg_by_cat.index.isin(needs_cats)].sum()
actual_wants   = avg_by_cat[avg_by_cat.index.isin(wants_cats)].sum()
actual_savings = max(salary - actual_needs - actual_wants, 0.0)
```

`avg_by_cat` is the average monthly spend per category across all loaded months. `max(..., 0.0)` clamps to zero — a negative "savings" (overspend) is shown via the warning in Section E rather than a negative bar.

**Per-month charts**

Before the average, individual bar charts are shown for each loaded month, three per row:

```python
COLS_PER_ROW = 3
for i in range(0, len(sorted_month_keys), COLS_PER_ROW):
    batch = sorted_month_keys[i:i + COLS_PER_ROW]
    cols  = st.columns(len(batch))
    for col, mk in zip(cols, batch):
        by_cat = month_data["outflow_df"].groupby("Category")["Amount"].sum().abs()
        mn = by_cat[by_cat.index.isin(needs_cats)].sum()
        mw = by_cat[by_cat.index.isin(wants_cats)].sum()
        ms = max(salary - mn - mw, 0.0)
        col.plotly_chart(bucket_bar_fig(mn, mw, ms, month_data["month_label"],
                                        show_legend=False), use_container_width=True)
```

Each month's savings is derived independently (`salary − that month's needs − that month's wants`), so months with unusually high spending visibly show a lower Savings bar. The legend is hidden on per-month charts to save horizontal space.

**Average breakdown**

After all month charts, the average across all months is shown with three `st.metric` cards (actual vs 50/30/20 target delta) and a grouped bar chart (Actual vs 50/30/20 Target) side by side.

---

### Section E — Long-Term Projection (Actual Savings Rate)

Uses `actual_savings` from Section D to drive the same `projection_chart` and `milestone_table` helpers as Section C — this time with the user's real figures instead of the hypothetical slider value:

```
salary  −  avg monthly needs spend  −  avg monthly wants spend  =  net saved / month
```

The caption makes this arithmetic explicit so the user knows exactly where the figure comes from. If `salary == 0` (not entered) or `actual_savings == 0` (spending equals or exceeds take-home), a descriptive message is shown instead of the chart.

---
## 17. Full flow diagram (current — modular structure)

### Module dependency map

```
app.py  (entry point)
│
├── config.py          ← BANK_FORMATS, DATE_FORMAT_OPTIONS
│
├── persistence.py     ← overrides dicts, save/load helpers
│   writes/reads:        inflow_overrides.json
│                        outflow_overrides.json
│                        categories.json
│                        bank_mappings.json
│
├── categories.py      ← categorize_transactions, add_keyword, propagate, delete
│   imports:             persistence.save_categories, outflow_overrides, make_tx_key
│
├── ingestion.py       ← make_fingerprint, detect_bank, normalise_df, load_transactions, get_month_info
│   imports:             config.BANK_FORMATS, categories.categorize_transactions
│
└── ui/
    ├── sidebar.py     ← render_sidebar(), delete_category()
    ├── welcome.py     ← show_welcome_guide()
    ├── mapping.py     ← show_mapping_ui()
    ├── month.py       ← render_month()
    ├── comparison.py  ← render_comparison()
    └── budget_rule.py ← render_5030_tab()
```

---

### Runtime flow (every Streamlit rerun)

```
App start
│
├─ st.set_page_config()           ← must be the very first Streamlit call
│
├─ import config, ingestion, persistence, ui.*   ← Python caches these after first import
│   └─ persistence.py module-level code runs once per Python process:
│       ├─ inflow_overrides  = load("inflow_overrides.json")   → dict in memory
│       └─ outflow_overrides = load("outflow_overrides.json")  → dict in memory
│
└─ main()
    │
    ├─ _apply_css()
    │   └─ @st.cache_data: load background image once, embed as base64 CSS
    │
    ├─ _init_session_state()
    │   ├─ if "categories" not in session_state:
    │   │   └─ load categories.json → st.session_state.categories
    │   └─ if "months" not in session_state:
    │       └─ st.session_state.months = {}
    │
    ├─ st.file_uploader(accept_multiple_files=True)
    │
    ├─ render_sidebar()   [ui/sidebar.py]
    │   ├─ Add Category → save_categories() → categories.json
    │   └─ Delete ✕    → delete_category()
    │                       ├─ del st.session_state.categories[cat]
    │                       ├─ save_categories()
    │                       └─ reset matching rows → "Uncategorized" in all months
    │
    ├─ if no files uploaded:
    │   ├─ st.session_state.months = {}
    │   ├─ show_welcome_guide()   [ui/welcome.py]
    │   └─ return
    │
    ├─ Cleanup: remove months whose source file was deselected
    │
    ├─ For each NEW file (not in processed_names):
    │   ├─ pd.read_csv → strip columns
    │   ├─ make_fingerprint()  [ingestion.py]
    │   ├─ detect_bank()       [ingestion.py]   ──found──▶ fmt = BANK_FORMATS[bank]
    │   │       │
    │   │    not found → check bank_mappings.json ──found──▶ fmt = saved custom mapping
    │   │       │
    │   │    not found → skip (mapping UI pass handles it)
    │   │
    │   ├─ load_transactions(file, fmt)   [ingestion.py]
    │   │   ├─ pd.read_csv
    │   │   ├─ normalise_df  → Date / Description / Amount
    │   │   │   └─ split columns: Amount = credit − debit  (HSBC-style)
    │   │   └─ categorize_transactions  [categories.py]  → + Category column
    │   │
    │   ├─ get_month_info(df)  → month_key ("2026-05"), month_label ("May 2026")
    │   │
    │   └─ st.session_state.months[month_key] = {
    │           "outflow_df": apply_outflow_overrides(df[Amount < 0])
    │           "inflow_df":  apply_inflow_overrides(df[Amount > 0])
    │       }
    │
    ├─ Mapping UI pass  [ui/mapping.py]
    │   └─ show_mapping_ui() → user picks columns → save bank_mappings.json → st.rerun()
    │
    ├─ Success banners: "Detected: Monzo — May 2026 loaded"
    │
    └─ st.tabs(["May 2026", ..., "Compare Months", "50/30/20 Rule"])
        │
        ├─ render_month(month_key)   [ui/month.py] ← one call per month tab
        │   ├─ Date range filter (guarded ±2 days)
        │   ├─ filter out / inf to selected range
        │   │
        │   ├─ sub-tab: Cash Outflow
        │   │   ├─ out_display = out.reset_index(drop=True)
        │   │   ├─ st.form
        │   │   │   ├─ st.data_editor (SelectboxColumn for Category)
        │   │   │   └─ st.form_submit_button "Apply Changes"
        │   │   └─ if submitted:
        │   │       ├─ for each changed row:
        │   │       │   ├─ outflow_df.at[real_idx] = new category
        │   │       │   ├─ add_keyword_to_category()  [categories.py]
        │   │       │   └─ propagate_category_to_all_months()  [categories.py]
        │   │       │       ├─ update all months' outflow_df in memory
        │   │       │       └─ write override key for every matching row
        │   │       ├─ save_outflow_overrides() → outflow_overrides.json
        │   │       ├─ st.session_state[msg_key] = changed_count
        │   │       └─ st.rerun()
        │   │
        │   ├─ sub-tab: Cash Inflow  (same pattern → inflow_overrides.json)
        │   └─ Expense summary + charts
        │
        ├─ render_comparison()   [ui/comparison.py]
        │
        └─ render_5030_tab()     [ui/budget_rule.py]
            │
            ├─ Section A: rule explanation (collapsed expander)
            │
            ├─ Section B: Budget Calculator
            │   ├─ st.number_input  → salary
            │   ├─ st.slider        → monthly_savings (default = 20%, max = 2×)
            │   └─ 3 st.metric cards: Needs / Wants / Savings % of take-home
            │
            ├─ Section C: Hypothetical Projection
            │   ├─ projection_chart(monthly_savings)  ← 0% / 3% / 7%, 500 months
            │   └─ milestone_table(monthly_savings)   ← value at 1/5/10/20/40 yr
            │
            ├─ Section D: Your Spending vs 50/30/20  (requires CSV upload)
            │   ├─ Needs multiselect  (options = available_cats − current_wants)
            │   ├─ Wants multiselect  (options = available_cats − needs_cats)
            │   ├─ Warning if any category is in neither bucket
            │   ├─ actual_savings = max(salary − needs_spend − wants_spend, 0)
            │   │
            │   ├─ Per-month charts (3 per row)
            │   │   └─ each: mn / mw / ms derived independently per month
            │   │
            │   └─ Average breakdown
            │       ├─ avg_by_cat = total per category / n_months
            │       ├─ 3 st.metric with delta vs 50/30/20 target
            │       └─ grouped bar chart: Actual vs 50/30/20 Target
            │
            └─ Section E: Long-Term Projection (Actual Rate)
                ├─ caption: salary − avg needs − avg wants = net saved/month
                ├─ projection_chart(actual_savings)
                └─ milestone_table(actual_savings)
```

---
## 18. Logic unchanged across modules

These functions and data structures were not rewritten during the modular restructure — only moved to the module that owns their responsibility:

| Item | Lives in | Purpose |
|---|---|---|
| `BANK_FORMATS` | `config.py` | Hardcoded column mappings for Revolut, Monzo, Barclays, HSBC, Starling |
| `DATE_FORMAT_OPTIONS` | `config.py` | Date format dropdown for the mapping UI |
| `make_fingerprint` | `ingestion.py` | Sorted column string for bank identification |
| `detect_bank` | `ingestion.py` | Hardcoded column heuristics |
| `load_bank_mappings` / `save_bank_mappings` | `persistence.py` | JSON read/write for custom bank mappings |
| `normalise_df` | `ingestion.py` | Maps bank-specific columns → `Date / Description / Amount` |
| `load_transactions` | `ingestion.py` | Wraps normalise + categorise |
| `categorize_transactions` | `categories.py` | Keyword-based auto-categorisation on CSV load |
| `add_keyword_to_category` | `categories.py` | Appends a description to a category's keyword list |
| `save_categories` | `persistence.py` | Writes `st.session_state.categories` to `categories.json` |
| `apply_inflow_overrides` / `apply_outflow_overrides` | `persistence.py` | Stamps saved category overrides onto a freshly loaded DataFrame |
| `make_tx_key` | `persistence.py` | `date\|description\|amount` string used as override dict key |
| All Plotly charts | `ui/month.py`, `ui/comparison.py`, `ui/budget_rule.py` | Pie, bar, line — logic unchanged |
| Expense summary table | `ui/month.py` | `groupby`, `merge`, `sort` — logic unchanged |
| CSS / background image | `app.py` | Frosted glass styling — unchanged |
| `render_comparison` | `ui/comparison.py` | Compare Months tab — logic unchanged |

---
## 19. Modular structure — how the app is organised

The app is split by **responsibility** into modules that can be read and understood independently. Each module owns one concern and nothing else.

```
app.py             ← entry point; wires everything together
config.py          ← constants (BANK_FORMATS, DATE_FORMAT_OPTIONS)
persistence.py     ← all file I/O (read/write JSON, override dicts)
categories.py      ← transaction categorisation logic
ingestion.py       ← CSV reading and normalisation
ui/
  sidebar.py       ← render_sidebar()
  welcome.py       ← show_welcome_guide()
  mapping.py       ← show_mapping_ui()
  month.py         ← render_month()
  comparison.py    ← render_comparison()
  budget_rule.py   ← render_5030_tab()
```

---

### `app.py` — entry point and orchestrator

`app.py` is the only file Streamlit runs directly (`streamlit run app.py`). Its job is to wire everything together, not to implement anything itself.

```python
# Must be the very first Streamlit call — no imports before this
st.set_page_config(page_title="Simple Finance Tracking App", page_icon="🤑", layout="wide")

# All other imports come after set_page_config
from config      import BANK_FORMATS
from ingestion   import make_fingerprint, detect_bank, load_transactions, get_month_info
from persistence import apply_inflow_overrides, apply_outflow_overrides, ...
from ui.mapping  import show_mapping_ui
...
```

**Why must `st.set_page_config` be first?**
Streamlit raises an error if any other Streamlit call (including those triggered by imports) runs before `set_page_config`. Putting all imports after it guarantees nothing else runs first.

Three functions live in `app.py`:

| Function | What it does |
|---|---|
| `_apply_css()` | Loads background image (cached with `@st.cache_data`), injects CSS |
| `_init_session_state()` | Restores `categories` and `months` from disk on new session |
| `main()` | Runs every rerun: CSS → session init → uploader → sidebar → process files → render tabs |

**`_init_session_state()` — why this exists**

```python
def _init_session_state():
    if "categories" not in st.session_state:
        st.session_state.categories = {"Uncategorized": [], "New category": []}
        if os.path.exists("categories.json"):
            with open("categories.json", "r") as f:
                st.session_state.categories = json.load(f)
    if "months" not in st.session_state:
        st.session_state.months = {}
```

Pressing F5 clears `st.session_state` completely. This function is called at the top of `main()` every rerun. The `if key not in session_state` guard means it only loads from disk on the *first* rerun of a new session — subsequent reruns skip it because the keys are already present.

**`@st.cache_data` on image loading**

```python
@st.cache_data
def _load_bg_image(path: str) -> str:
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()
```

Without `@st.cache_data`, Python would read and base64-encode the image file on *every* Streamlit rerun. The decorator caches the return value after the first call; subsequent calls return the cached string without touching the disk.

---

### `config.py` — constants only

```python
BANK_FORMATS = {
    "Monzo":   {"date_col": "Date", "description_col": "Name", "amount_col": "Amount"},
    "Revolut": {...},
    ...
}
DATE_FORMAT_OPTIONS = {"%d/%m/%Y": "DD/MM/YYYY", "%Y-%m-%d": "YYYY-MM-DD", ...}
```

No functions, no imports, no Streamlit. Just two dictionaries.

`BANK_FORMATS` is referenced by both `ingestion.py` (for loading) and `ui/mapping.py` (for the mapping UI). A separate `config.py` avoids a circular import — if `BANK_FORMATS` lived in `ingestion.py`, then `ui/mapping.py` would need to import `ingestion.py`, which imports `categories.py`, creating a chain.

---

### `persistence.py` — all file I/O in one place

```python
# Module-level: loaded ONCE per Python process, shared by all modules
inflow_overrides  = _load_json("inflow_overrides.json")
outflow_overrides = _load_json("outflow_overrides.json")
```

These dicts are loaded at import time — once when Streamlit starts, not once per rerun. All modules that import them share the same object in memory. When `categories.py` adds a key, `app.py` sees it immediately.

**Functions in `persistence.py`:**

| Function | Purpose |
|---|---|
| `make_tx_key(row)` | `"date\|description\|amount"` — unique key per transaction |
| `apply_inflow_overrides(df)` | Stamps saved categories onto inflow rows |
| `apply_outflow_overrides(df)` | Stamps saved categories onto outflow rows |
| `save_categories()` | Writes `st.session_state.categories` → `categories.json` |
| `save_inflow_overrides()` | Writes in-memory dict → `inflow_overrides.json` |
| `save_outflow_overrides()` | Writes in-memory dict → `outflow_overrides.json` |
| `load_bank_mappings()` | Reads `bank_mappings.json` |
| `save_bank_mappings(mappings)` | Writes `bank_mappings.json` |

---

### `categories.py` — transaction categorisation logic

| Function | What it does |
|---|---|
| `categorize_transactions(df)` | Assigns a category to each row by matching `Description` against keyword lists in `st.session_state.categories` |
| `add_keyword_to_category(cat, desc)` | Appends a description to a category's keyword list; saves to `categories.json` |
| `propagate_category_to_all_months(desc, cat)` | Updates every loaded month's `outflow_df` and writes override keys for all matching rows |
| `delete_category(cat_name)` | Removes a category, saves, and resets all matching rows to Uncategorized |

---

### `ingestion.py` — reading and normalising bank CSVs

```python
def make_fingerprint(columns) -> str:
    return "|".join(sorted(col.strip().lower() for col in columns))

def detect_bank(columns) -> str | None: ...
    # Returns the bank name if columns match a known signature, else None

def normalise_df(raw_df, fmt) -> pd.DataFrame: ...
    # Maps bank-specific column names → Date / Description / Amount
    # Handles split debit/credit columns: Amount = credit − debit

def load_transactions(file, fmt) -> pd.DataFrame | None: ...
    # normalise_df → categorize_transactions → returns full DataFrame

def get_month_info(df) -> tuple[str, str]: ...
    # Returns ("2026-05", "May 2026") using dominant month in the data
```

---

### `ui/` — one file per visual section

Each file exports exactly one top-level function called by `app.py`:

| File | Exports | Renders |
|---|---|---|
| `ui/sidebar.py` | `render_sidebar()` | Category management sidebar |
| `ui/welcome.py` | `show_welcome_guide()` | 4-step onboarding shown before upload |
| `ui/mapping.py` | `show_mapping_ui(raw_df, suffix)` | Column mapping UI for unknown bank formats |
| `ui/month.py` | `render_month(month_key)` | Full month tab: date filter, editors, charts |
| `ui/comparison.py` | `render_comparison()` | Compare Months tab |
| `ui/budget_rule.py` | `render_5030_tab()` | 50/30/20 Rule tab (5 sections) |

---

### `requirements.txt` and `.gitignore`

**`requirements.txt`** — three packages needed to run the app:
```
streamlit>=1.35
pandas>=2.0
plotly>=5.20
```

**`.gitignore`** — excludes files that should not be committed:
- `__pycache__/` — Python bytecode (auto-generated)
- `*.csv` — personal bank statements (private financial data)
- `categories.json`, `bank_mappings.json`, `inflow_overrides.json`, `outflow_overrides.json` — user-specific runtime state
- `.venv/`, `venv/` — virtual environments
- `.ipynb_checkpoints/` — Jupyter auto-save files

---
## 20. Bucket mutual exclusion — Needs/Wants multiselects

### The problem

In Section D of the 50/30/20 tab, the user assigns categories to two buckets: **Needs** and **Wants**. Without any guards, the original implementation allowed the same category to be selected in both multiselects simultaneously. If "Groceries" appeared in both Needs and Wants, it would be double-counted — adding to both totals and producing an inflated spend figure that exceeded the salary.

There was also no prompt to remind the user that some categories had been left unassigned, meaning the Savings figure would be silently inflated (because unassigned spending was excluded from Needs and Wants but still came out of take-home pay).

---

### The solution — cross-filtering options

Each multiselect's available choices exclude whatever the other bucket already has selected:

```python
available_cats = [c for c in st.session_state.categories if c != "Uncategorized"]

# Step 1: compute Needs options — remove anything already chosen in Wants
needs_options = [c for c in available_cats if c not in st.session_state.get("5030_wants", [])]

col1, col2 = st.columns(2)
with col1:
    st.markdown("**Needs** — essentials")
    needs_cats = st.multiselect(
        "needs_select", needs_options, key="5030_needs",
        label_visibility="collapsed",
    )
with col2:
    # Step 2: compute Wants options — remove what was JUST returned from Needs
    wants_options = [c for c in available_cats if c not in needs_cats]
    st.markdown("**Wants** — lifestyle")
    wants_cats = st.multiselect(
        "wants_select", wants_options, key="5030_wants",
        label_visibility="collapsed",
    )
```

---

### Why there is an asymmetry between Needs and Wants

Needs options use `st.session_state.get("5030_wants", [])` — the *previous* rerun's value.  
Wants options use `needs_cats` — the *current* rerun's return value.

This asymmetry is necessary because of how Streamlit's script execution model works:

```
User clicks "Groceries" in Needs multiselect
        │
        ▼
Streamlit reruns the script from top to bottom
        │
        ├─ st.session_state["5030_needs"] = ["Groceries"]   ← already updated
        │  st.session_state["5030_wants"] = ["Dining"]      ← unchanged from last rerun
        │
        ├─ needs_options = available_cats − ["Dining"]       ← excludes Wants correctly ✓
        │
        ├─ Needs multiselect renders → returns ["Groceries"]
        │  needs_cats = ["Groceries"]
        │
        └─ wants_options = available_cats − ["Groceries"]   ← excludes Needs immediately ✓
           (using needs_cats, the return value from THIS rerun)

Result: both multiselects updated correctly in a single rerun, with no lag.
```

If Wants options were also computed from session state (the previous rerun's `"5030_needs"` value), there would be a one-rerun lag: the user would see "Groceries" disappear from Wants only on the *next* interaction, not immediately. Using the return value of the Needs multiselect eliminates that lag.

---

### Unassigned categories warning

After both multiselects render, the code checks which categories appear in neither bucket:

```python
unassigned = [c for c in available_cats if c not in needs_cats and c not in wants_cats]
if unassigned:
    st.warning(
        f"Not yet assigned to Needs or Wants: **{', '.join(unassigned)}**. "
        "Assign all categories for an accurate breakdown."
    )
```

**Why this matters:**  
`actual_savings = max(salary − actual_needs − actual_wants, 0.0)`

If "Transport" (£200/month) is left unassigned, the Needs total is understated by £200, and the derived Savings figure is overstated by £200. The warning prevents the user from acting on misleading numbers without realising why.

The warning is informational — it does not block the breakdown from rendering. The user might legitimately leave "Uncategorized" transactions out of the rule for a period while they finish categorising.

---

### How new categories are handled automatically

When the user adds a new category via the sidebar, it appears in `st.session_state.categories`. On the next rerun, `available_cats` is recomputed and the new category appears in both multiselects' option lists — it is automatically visible without any code change needed. The user simply selects it into whichever bucket applies.

Because the new category starts in neither bucket, it immediately triggers the "not yet assigned" warning, prompting the user to place it before closing the tab.